# SEAL Subgraph-Based Link Prediction

Link Prediction on Cora (Planetoid): Enclosing subgraph extraction and node labeling for accurate link prediction. This notebook implements the approach with `DGCNN / SortAggregation` inside a `K3SEALNet` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `DGCNN / SortAggregation` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "SEAL Subgraph-Based Link Prediction"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. DGCNN Subgraph Classifier Model for SEAL
class K3SEALNet(keras.Model):
    def __init__(self, in_channels, hidden_channels=32):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, hidden_channels)
        self.lin1 = layers.Dense(64, activation="relu")
        self.lin2 = layers.Dense(1)

    def call(self, x, edge_index, batch=None):
        x = ops.relu(self.conv1(x, edge_index))
        x = ops.relu(self.conv2(x, edge_index))
        out = k3_layers.global_mean_pool(x, batch)
        out = self.lin1(out)
        return ops.squeeze(self.lin2(out), axis=-1)

k3_model = K3SEALNet(in_channels=16)

# 2. Forward pass test
num_nodes = 40
dummy_x = keras.random.normal((num_nodes, 16))
dummy_edges = ops.convert_to_tensor([[0, 1, 2], [1, 2, 3]], dtype="int64")
dummy_batch = ops.zeros((num_nodes,), dtype="int64")

out = k3_model(dummy_x, dummy_edges, dummy_batch)
print(f"SEAL predicted link probability logit: {float(out[0]):.4f}")

print("\n✓ K3-Node SEAL execution completed successfully!")